<a href="https://colab.research.google.com/github/nikhil-arora88/ABTesting/blob/master/Text_classification_VerteXAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
! pip3 install --upgrade --quiet google-cloud-aiplatform \
                                    google-cloud-storage \
                                    jsonlines

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 66.6 MB/s eta 0:00:00


In [8]:
# Automatically restart kernel after installs so that your environment can access the new packages
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
PROJECT_ID = "project-d6bb8c56-278c-47bb-993"  # @param {type:"string"}

# set the project id
! gcloud config set project $PROJECT_ID

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [2]:
REGION = "us-central1"  # @param {type: "string"}

In [3]:
from google.colab import auth
auth.authenticate_user()

In [4]:
BUCKET_NAME = "vertexai-text-classification-nikhil"  # @param {type:"string"}
BUCKET_URI = f"gs://{BUCKET_NAME}"
! gsutil mb -l $REGION $BUCKET_URI

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Creating gs://vertexai-text-classification-nikhil/...
ServiceException: 409 A Cloud Storage bucket named 'vertexai-text-classification-nikhil' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


In [5]:
import jsonlines
from google.cloud import aiplatform, storage
from google.cloud.aiplatform import jobs

Create a Dataset resource and import your data

In [6]:
aiplatform.init(project=PROJECT_ID, location=REGION)

In [7]:
# # Use a timestamp to ensure unique resources
# src_uris = "gs://vertexai-dev-text-classification/data_happiness.csv"
# display_name = "text-classification-dataset"

# text_dataset = aiplatform.TextDataset.create(
#     display_name=display_name,
#     gcs_source=src_uris,
#     import_schema_uri=aiplatform.schema.dataset.ioformat.text.single_label_classification,
#     sync=True,
# )

Access Dataset resource with Dateset_id

In [8]:
text_dataset = aiplatform.TabularDataset('3470040555319197696')

Create a Training job

In [9]:
# training_job_display_name = "text-classification-model"
# job = aiplatform.AutoMLTextTrainingJob(
#     display_name=training_job_display_name,
#     prediction_type="classification",
#     multi_label=False,
# )

Run Training Job and Create the model

In [10]:
# model_display_name = "text-classification-model"

# # Run the training job
# model = job.run(
#     dataset=text_dataset,
#     model_display_name=model_display_name,
#     training_fraction_split=0.8,
#     validation_fraction_split=0.1,
#     test_fraction_split=0.1,
#     sync=True,
# )

Access Model with model-id

In [11]:
model = aiplatform.Model("projects/project-d6bb8c56-278c-47bb-993/locations/us-central1/models/1594976856019304448")

Deploy your model to an endpoint for online prediction

In [12]:
# deployed_model_display_name = "text-classification-model"

# endpoint = model.deploy(
#     deployed_model_display_name=deployed_model_display_name, sync=True
# )

In [13]:
endpoint = aiplatform.Endpoint('656804815724085248')

Get Online prediction

In [14]:
content = "I like hanging out with my friends"

response = endpoint.predict(
    instances=[
        {"Text": content}
    ]
)

for pred in response.predictions:
    classes = pred["classes"]
    scores = pred["scores"]

    # Print all class probabilities
    for cls, score in zip(classes, scores):
        print(f"{cls}: {score:.6f}")

    # Get top prediction
    top_idx = scores.index(max(scores))
    print("\nTop Prediction:")
    print(f"Class: {classes[top_idx]}")
    print(f"Confidence: {scores[top_idx]:.6f}")

affection: 0.004620
achievement: 0.000019
enjoy_the_moment: 0.000014
bonding: 0.995342
leisure: 0.000001
nature: 0.000002
exercise: 0.000002

Top Prediction:
Class: bonding
Confidence: 0.995342


Create Batch Prediction

In [15]:
instances = [
    "We hiked through the woods and up the hill to the ice caves",
    "My kitten is so cute",
]
input_file_name = "batch-prediction-input.jsonl"

In [16]:
# Instantiate the Storage client and create the new bucket
# from google.cloud import  storage
storage_client = storage.Client()
bucket = storage_client.get_bucket(BUCKET_NAME)
# Iterate over the prediction instances, creating a new TXT file
# for each.
input_file_data = []
for count, instance in enumerate(instances):
    instance_name = f"input_{count}.txt"
    instance_file_uri = f"{BUCKET_URI}/{instance_name}"
    # Add the data to store in the JSONL input file.
    tmp_data = {"content": instance_file_uri, "mimeType": "text/plain"}
    input_file_data.append(tmp_data)

    # Create the new instance file
    blob = bucket.blob(instance_name)
    blob.upload_from_string(instance)

input_str = "\n".join([str(d) for d in input_file_data])
file_blob = bucket.blob(f"{input_file_name}")

file_blob.upload_from_string(input_str)

In [17]:
bucket

<Bucket: vertexai-text-classification-nikhil>

In [18]:
job_display_name = "e2e-text-classification-batch-prediction-job"
# model = aiplatform.Model(model_name=model.name)
batch_prediction_job = model.batch_predict(
    job_display_name=job_display_name,
    gcs_source="gs://vertexai-text-classification-nikhil/batch-prediction-input.jsonl",
    gcs_destination_prefix=f"{BUCKET_URI}/output",
    sync=True,
)
batch_prediction_job_name = batch_prediction_job.resource_name

RuntimeError: Job failed with:
code: 3
message: "Job failed. See logs for full details."


Clean UP

In [ ]:
delete_bucket = False

if delete_bucket or os.getenv("IS_TESTING"):
    ! gsutil rm -r $BUCKET_URI

# Delete batch
batch_job.delete()

# Undeploy endpoint
endpoint.undeploy_all()

# `force` parameter ensures that models are undeployed before deletion
endpoint.delete()

# Delete model
model.delete()

# Delete text dataset
text_dataset.delete()

# Delete training job
job.delete()